In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


In [2]:

# ============================================================
# 1) Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [3]:

# ============================================================
# 2) Files to test
# ============================================================
DATASETS = [
    "../fraud_preprocessing/fraud_prepared_numeric.csv",
    "../wrapper/fraud_selected_features_rfecv.csv",
    "../PCA/fraud_pca_95_variance.csv"
]

TARGET_COL = "fraud"


In [4]:

# ============================================================
# 3) Build MLP model
# ============================================================
def build_mlp_model(input_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),

        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="roc_auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc")
        ]
    )
    return model


In [5]:

# ============================================================
# 4) Tune threshold on validation set
# ============================================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.05)
    best_threshold = 0.50
    best_f1 = -1

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = thr

    return best_threshold, best_f1


In [6]:

# ============================================================
# 5) Train and evaluate one dataset
# ============================================================
def run_experiment(csv_path, target_col="fraud", epochs=30, batch_size=256):
    print("=" * 80)
    print(f"DATASET: {csv_path}")

    if not os.path.exists(csv_path):
        print(f"File not found: {csv_path}")
        print("Skipping...\n")
        return None

    # ----------------------------
    # Load data
    # ----------------------------
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}")

    # Fill missing feature values if any
    for col in df.columns:
        if col != target_col and df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].astype(int).copy()

    print(f"Shape: {df.shape}")
    print(f"Features: {X.shape[1]}")
    print("Class distribution:")
    print(y.value_counts(normalize=True).rename("proportion"))

    # ----------------------------
    # Split: train / val / test
    # ----------------------------
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y,
        test_size=0.20,
        stratify=y,
        random_state=SEED
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=SEED
    )

    # ----------------------------
    # Scale features
    # ----------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # ----------------------------
    # Class weights
    # ----------------------------
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    print("Class weights:", class_weight_dict)

    # ----------------------------
    # Model
    # ----------------------------
    model = build_mlp_model(input_dim=X_train_scaled.shape[1])
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1
        )
    ]

    # ----------------------------
    # Train
    # ----------------------------
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    # ----------------------------
    # Validation threshold tuning
    # ----------------------------
    val_prob = model.predict(X_val_scaled, verbose=0).ravel()
    best_threshold, best_val_f1 = find_best_threshold(y_val, val_prob)

    print(f"Best validation threshold: {best_threshold:.2f}")
    print(f"Best validation F1: {best_val_f1:.4f}")

    # ----------------------------
    # Test evaluation
    # ----------------------------
    test_prob = model.predict(X_test_scaled, verbose=0).ravel()
    test_pred = (test_prob >= best_threshold).astype(int)

    metrics = {
        "dataset": csv_path,
        "n_features": X.shape[1],
        "best_threshold": best_threshold,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, test_prob),
        "pr_auc": average_precision_score(y_test, test_prob)
    }

    cm = confusion_matrix(y_test, test_pred)

    print("\nTest Metrics")
    for k, v in metrics.items():
        if k not in ["dataset", "n_features"]:
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    print("\nConfusion Matrix")
    print(cm)

    # ----------------------------
    # Save outputs
    # ----------------------------
    base_name = os.path.splitext(os.path.basename(csv_path))[0]

    model_dir = os.path.join("..", "model", "MLP")
    csv_dir = os.path.join(model_dir, "csv")

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)

    history_path = os.path.join(csv_dir, f"{base_name}_mlp_history.csv")
    pred_path = os.path.join(csv_dir, f"{base_name}_mlp_test_predictions.csv")
    model_path = os.path.join(model_dir, f"{base_name}_mlp_model.keras")

    pd.DataFrame(history.history).to_csv(history_path, index=False)

    pd.DataFrame({
        "y_true": y_test.reset_index(drop=True),
        "y_prob": test_prob,
        "y_pred": test_pred
    }).to_csv(pred_path, index=False)

    model.save(model_path)

    print("\nSaved:")
    print(f"- {history_path}")
    print(f"- {pred_path}")
    print(f"- {model_path}")

    return metrics


In [7]:

# ============================================================
# 6) Run on all datasets
# ============================================================
all_results = []

for file_name in DATASETS:
    result = run_experiment(file_name, target_col=TARGET_COL, epochs=30, batch_size=256)
    if result is not None:
        all_results.append(result)


DATASET: ../fraud_preprocessing/fraud_prepared_numeric.csv


Shape: (180519, 56)
Features: 55
Class distribution:
fraud
0    0.977498
1    0.022502
Name: proportion, dtype: float64


Class weights: {0: 0.5115113519640138, 1: 22.217692307692307}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           7,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

...

452/452 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5419 - loss: 0.6872 - pr_auc: 0.0277 - precision: 0.0270 - recall: 0.5515 - roc_auc: 0.5622 - val_accuracy: 0.5479 - val_loss: 0.6869 - val_pr_auc: 0.0261 - val_precision: 0.0247 - val_recall: 0.4969 - val_roc_auc: 0.5417 - learning_rate: 6.2500e-05


Epoch 19: early stopping


Restoring model weights from the end of the best epoch: 14.


Best validation threshold: 0.50
Best validation F1: 0.0479



Test Metrics
dataset: ../PCA/fraud_pca_95_variance.csv
n_features: 11
best_threshold: 0.5000
accuracy: 0.5490
precision: 0.0239
recall: 0.4778
f1: 0.0455
roc_auc: 0.5232
pr_auc: 0.0250

Confusion Matrix
[[19434 15858]
 [  424   388]]

Saved:
- ..\model\MLP\csv\fraud_pca_95_variance_mlp_history.csv
- ..\model\MLP\csv\fraud_pca_95_variance_mlp_test_predictions.csv
- ..\model\MLP\fraud_pca_95_variance_mlp_model.keras


In [8]:

# ============================================================
# 7) Compare results
# ============================================================
if len(all_results) > 0:
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(by="pr_auc", ascending=False)

    print("\n" + "=" * 80)
    print("FINAL COMPARISON")
    print(results_df.to_string(index=False))

    model_dir = os.path.join("..", "model", "MLP")
    csv_dir = os.path.join(model_dir, "csv")

    results_df.to_csv(
    os.path.join(csv_dir, "mlp_results_comparison.csv"),
    index=False
    )
    print("\nSaved: mlp_results_comparison.csv")
else:
    print("No dataset was successfully processed.")


FINAL COMPARISON
                                          dataset  n_features  best_threshold  accuracy  precision   recall       f1  roc_auc   pr_auc
../fraud_preprocessing/fraud_prepared_numeric.csv          55            0.65  0.923499   0.045243 0.119458 0.065629 0.610418 0.034310
     ../wrapper/fraud_selected_features_rfecv.csv          17            0.50  0.664525   0.025370 0.371921 0.047499 0.534657 0.025336
                 ../PCA/fraud_pca_95_variance.csv          11            0.50  0.549025   0.023883 0.477833 0.045492 0.523221 0.025045

Saved: mlp_results_comparison.csv
